# SKU Mapping Pipeline
**Sources**: Maxab products (Snowflake) + Competitor scraped data (Snowflake) + Existing manual mapping (Snowflake)

**Approach**: Validate existing mapping → Run algorithm on unmapped → Combine

In [2]:
!pip install rapidfuzz wn openpyxl pandas numpy snowflake-connector-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.9/57.9 MB 76.2 MB/s  0:00:00 eta 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
  Created wheel for rapidfuzz: filename=rapidfuzz-3.14.3-py3-none-any.whl size=64274 sha256=1129ec38de790b99cc8d68636d868b2b3027bbb568f901dbfad8d9385e8f7937
  Stored in directory: /home/ec2-user/.cache/pip/wheels/4f/b0/18/3cba2cfa8fd2a6d1148f3f1dacd2e090692257cecc99613f17
Successfully built rapidfuzz
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [wn] 3/4 [wn]


In [3]:
import wn
wn.download('omw-arb:1.4')

Download [##############################] (599220/599220 bytes) Completeg
Read [##############################] (83267/83267) 
Added omw-arb:1.4 (Arabic WordNet (AWN v2))267/101267) Examplesonsionssours



PosixPath('/home/ec2-user/.wn_data/downloads/4cb8d2182ddb97c9a9d865fe4b0fd7c7a74d6a21')

In [5]:
import pandas as pd
import numpy as np
import re
import os
import warnings
from rapidfuzz import fuzz, process
from collections import Counter
from datetime import datetime
import wn
import snowflake.connector
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('__file__'), '..')))
sys.path.insert(0, r'd:\scripts\Mustafa')
sys.path.insert(0, r'/home/ec2-user/SageMaker/scripts/Mustafa')
import setup_environment_2

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 25)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 220)

setup_environment_2.initialize_env()

def query_snowflake(query):
    con = snowflake.connector.connect(
        user=os.environ["SNOWFLAKE_USERNAME"],
        account=os.environ["SNOWFLAKE_ACCOUNT"],
        password=os.environ["SNOWFLAKE_PASSWORD"],
        database=os.environ["SNOWFLAKE_DATABASE"]
    )
    try:
        cur = con.cursor()
        cur.execute("USE WAREHOUSE COMPUTE_WH")
        cur.execute(query)
        data = cur.fetchall()
        columns = [desc[0] for desc in cur.description]
        return pd.DataFrame(data, columns=columns)
    finally:
        con.close()

print("Environment initialized.")

/home/ec2-user/.Renviron
/home/ec2-user/service_account_key.json
Environment initialized.


## Load Data from Snowflake

In [9]:
MAXAB_QUERY = open(r'current_sku_data.sql').read()
SCRAPPED_QUERY = open(r'raw_scraped_data_latest.sql').read()
EXISTING_MAPPING_QUERY = open(r'existing_mapping_query.sql').read()

print("Loading maxab products...")
maxab = query_snowflake(MAXAB_QUERY)
maxab.columns = [c.upper() for c in maxab.columns]
print(f"  Maxab: {maxab.shape[0]} rows, {maxab['PRODUCT_ID'].nunique()} unique products")

print("Loading scrapped data...")
scrapped = query_snowflake(SCRAPPED_QUERY)
scrapped.columns = [c.upper() for c in scrapped.columns]
print(f"  Scrapped: {scrapped.shape[0]} rows from {scrapped['SOURCE_APP'].nunique()} apps")

print("Loading existing manual mapping...")
existing_map = query_snowflake(EXISTING_MAPPING_QUERY)
existing_map.columns = [c.upper() for c in existing_map.columns]
print(f"  Existing mapping: {existing_map.shape[0]} rows, {existing_map['MAXAB_PRODUCT_ID'].nunique()} unique maxab products")

print(f"\nScrapped per app:\n{scrapped['SOURCE_APP'].value_counts().to_string()}")
print(f"\nExisting mapping per app:\n{existing_map['SOURCE_APP'].value_counts().to_string()}")

Loading maxab products...
  Maxab: 5982 rows, 3811 unique products
Loading scrapped data...
  Scrapped: 7435 rows from 4 apps
Loading existing manual mapping...
  Existing mapping: 4792 rows, 2236 unique maxab products

Scrapped per app:
SOURCE_APP
Cartona    2775
Tawfeer    2267
Speed      1333
Talabia    1060

Existing mapping per app:
SOURCE_APP
Cartona    1988
Talabia    1064
Tawfeer     921
Speed       819


## Arabic NLP Core (normalization, size/count extraction, AWN synonyms)

In [15]:
ARABIC_DIACRITICS = re.compile(r'[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06DC\u06DF-\u06E4\u06E7\u06E8\u06EA-\u06ED]')
ALEF_VARIANTS = {'\u0622': '\u0627', '\u0623': '\u0627', '\u0625': '\u0627', '\u0671': '\u0627'}
MEASUREMENT_NORMALIZATION = {
    'جرام': 'جم', 'غرام': 'جم', 'غم': 'جم', 'جرم': 'جم', 'g': 'جم', 'gm': 'جم', 'gr': 'جم',
    'كيلو': 'كجم', 'كيلوجرام': 'كجم', 'كيلوغرام': 'كجم', 'kg': 'كجم',
    'ملل': 'مل', 'ملي': 'مل', 'ميلي': 'مل', 'ml': 'مل',
    'لتر': 'لتر', 'ليتر': 'لتر', 'l': 'لتر', 'سنتي': 'سم', 'سنتيمتر': 'سم', 'cm': 'سم',
    'قطعه': 'قطعة', 'حبه': 'حبة', 'عبوه': 'عبوة', 'فتله': 'فتلة', 'بكره': 'بكرة', 'حفاضه': 'حفاضة',
}
SIZE_TO_BASE = {'جم': ('weight', 1.0), 'كجم': ('weight', 1000.0), 'مل': ('volume', 1.0), 'لتر': ('volume', 1000.0), 'سم': ('length', 10.0)}
COUNT_UNITS = set(['فتله','فتلة','بكره','بكرة','قطعه','قطعة','حبه','حبة','كيس','ظرف','رول',
                   'باكت','منديل','فوطه','فوطة','شريط','قرص','ورقه','ورقة','كبسوله','كبسولة','باكيت'])
EASTERN_TO_WESTERN = str.maketrans('٠١٢٣٤٥٦٧٨٩', '0123456789')

def normalize_arabic(t):
    if not isinstance(t, str) or not t.strip(): return ''
    t = ARABIC_DIACRITICS.sub('', t.strip()).replace('\u0640', '')
    for v, c in ALEF_VARIANTS.items(): t = t.replace(v, c)
    t = t.replace('\u0629', '\u0647').replace('\u0649', '\u064A').translate(EASTERN_TO_WESTERN).lower()
    t = re.sub(r'[^\w\s\u0600-\u06FF]', ' ', t)
    t = re.sub(r'(\d)([\u0600-\u06FF])', r'\1 \2', t)
    t = re.sub(r'([\u0600-\u06FF])(\d)', r'\1 \2', t)
    tokens = [MEASUREMENT_NORMALIZATION.get(x, x) for x in t.split()]
    return re.sub(r'\s+', ' ', ' '.join(tokens)).strip()

def strip_al(token):
    if token.startswith('ال') and len(token) > 2: return token[2:]
    return token

def extract_size_info(t): return re.findall(r'(\d+(?:\.\d+)?)\s*(كجم|جم|مل|لتر|سم)', normalize_arabic(t))
def normalize_count_unit(u): return u.replace('\u0629', '\u0647')
def extract_count_info(t):
    normalized = normalize_arabic(t)
    matches = []; seen = set()
    for unit in COUNT_UNITS:
        for m in re.finditer(r'(\d+)\s*' + re.escape(unit), normalized):
            key = (m.group(1), normalize_count_unit(unit))
            if key not in seen: matches.append((float(m.group(1)), normalize_count_unit(unit))); seen.add(key)
        for m in re.finditer(re.escape(unit) + r'\s*(\d+)', normalized):
            key = (m.group(1), normalize_count_unit(unit))
            if key not in seen: matches.append((float(m.group(1)), normalize_count_unit(unit))); seen.add(key)
    return matches

def normalize_size_to_base(v, u):
    if u in SIZE_TO_BASE: d, m = SIZE_TO_BASE[u]; return d, float(v) * m
    return None, None

def sizes_compatible(a, b, tol=0.15):
    sa, sb = extract_size_info(a), extract_size_info(b)
    ca, cb = extract_count_info(a), extract_count_info(b)
    if sa and sb:
        for va, ua in sa:
            da, ba = normalize_size_to_base(va, ua)
            if da is None: continue
            for vb, ub in sb:
                db, bb = normalize_size_to_base(vb, ub)
                if db is None: continue
                if da != db: return False
                mx = max(ba, bb)
                if mx > 0: return abs(ba - bb) / mx <= tol
                return True
    if ca and cb:
        for val_a, unit_a in ca:
            for val_b, unit_b in cb:
                if unit_a == unit_b:
                    mx = max(val_a, val_b)
                    if mx > 0: return abs(val_a - val_b) / mx <= tol
                    return True
    has_size_a, has_size_b = bool(sa), bool(sb)
    has_count_a, has_count_b = bool(ca), bool(cb)
    if (has_size_a and has_count_b and not has_size_b) or (has_count_a and has_size_b and not has_count_b):
        return False
    return None

# Arabic WordNet synonym cache
_awn_cache = {}
def get_awn_synonyms(word):
    if word in _awn_cache: return _awn_cache[word]
    result = set()
    try:
        for form in [word, strip_al(word)]:
            for ss in wn.synsets(form, lang='arb'):
                for w in ss.words(): result.add(ARABIC_DIACRITICS.sub('', w.lemma()))
    except: pass
    _awn_cache[word] = result
    return result

def text_sim(a, b):
    if not a or not b: return 0.0
    return max(fuzz.token_sort_ratio(a, b), fuzz.token_set_ratio(a, b) * 0.95, fuzz.partial_ratio(a, b) * 0.85)

def pct_diff(a, b):
    mn = min(a, b)
    if mn <= 0: return 1.0
    return (max(a, b) - mn) / mn

print("Arabic NLP core loaded.")

Arabic NLP core loaded.


## Preprocessing + Lookups

In [16]:
UNIT_TYPE_CANONICAL = {
    'carton': 'carton', 'كرتونه': 'carton', 'كرتون': 'carton', 'box': 'carton',
    'shrink': 'shrink', 'شرينك': 'shrink', 'شرنك': 'shrink', 'sherink': 'shrink',
    'علبه': 'box', 'item': 'item', 'عبوه': 'item', 'piece': 'piece', 'قطعه': 'piece',
    'pala': 'bala', 'باله': 'bala', 'packet': 'packet', 'باكت': 'packet',
    'jar': 'jar', 'برطمان': 'jar', 'kees': 'kees', 'كيس': 'kees', 'bag': 'kees',
    'jerry can': 'jerrycan', 'جركن': 'jerrycan', 'roll': 'roll', 'رول': 'roll',
    'shikara': 'shikara', 'shekara': 'shikara', 'شيكاره': 'shikara', 'شكاره': 'shikara',
    'strip': 'strip', 'شريط': 'strip', 'mat': 'mat', 'حصيره': 'mat',
    'cans': 'cans', 'can': 'cans', 'كانز': 'cans',
    'offer': 'offer', 'عرض': 'offer', 'تجميعه': 'offer',
    '1/2 carton': 'half_carton', 'نص كرتونه': 'half_carton',
    '1/4 carton': 'quarter_carton', 'ربع كرتونه': 'quarter_carton',
    '4/3 carton': '3q_carton', 'تلت ارباع كرتونه': '3q_carton',
    'pallet': 'pallet', 'بالته': 'pallet', 'rabta': 'rabta', 'ربطه': 'rabta',
    'lafa': 'lafa', 'لفه': 'lafa', 'mold': 'mold', 'قالب': 'mold',
    'tiplate': 'tiplate', 'صفيح': 'tiplate', 'plate': 'tiplate',
    'bottle': 'bottle', 'ازازه': 'bottle', 'bucket': 'bucket', 'جردل': 'bucket',
    'scratch card': 'scratch_card', 'كارت': 'scratch_card',
    'dozen': 'dozen', 'دسته': 'dozen', 'kg': 'kg',
    'نص شرينك': 'half_shrink', 'ربع شرينك': 'quarter_shrink', '10 كارت': 'scratch_card_10', 'شنطه': 'bag_large',
}
def norm_unit(ut):
    if not isinstance(ut, str): return 'unknown'
    return UNIT_TYPE_CANONICAL.get(normalize_arabic(ut.strip().lower()), normalize_arabic(ut.strip().lower()))

NO_QTY_APPS = {'Speed', 'Cartona'}
SOFT_THRESHOLD = 90
PRICE_HARD_MAX = 0.30

# Ensure numeric columns are numeric (Snowflake may return strings)
for col in ['CURRENT_PRICE', 'BASIC_UNIT_COUNT', 'PRODUCT_ID', 'PACKING_UNIT_ID']:
    if col in maxab.columns:
        maxab[col] = pd.to_numeric(maxab[col], errors='coerce')
for col in ['SCRAPPED_PRICE', 'QUANTITY_PER_UNIT']:
    if col in scrapped.columns:
        scrapped[col] = pd.to_numeric(scrapped[col], errors='coerce')
if 'MAXAB_PRODUCT_ID' in existing_map.columns:
    existing_map['MAXAB_PRODUCT_ID'] = pd.to_numeric(existing_map['MAXAB_PRODUCT_ID'], errors='coerce')

maxab['unit_canonical'] = maxab['PACKING_UNIT_NAME_AR'].apply(norm_unit)
scrapped['unit_canonical'] = scrapped['UNIT_TYPE'].apply(norm_unit)
maxab['sku_norm'] = maxab['SKU'].apply(normalize_arabic)
maxab['brand_norm'] = maxab['BRAND'].apply(normalize_arabic)
maxab['cat_norm'] = maxab['CAT'].apply(normalize_arabic)
scrapped['product_norm'] = scrapped['SCRAPPED_PRODUCT'].apply(normalize_arabic)
scrapped['brand_norm'] = scrapped['SCRAPPED_BRAND'].apply(normalize_arabic)

maxab_brands = sorted(maxab['brand_norm'].unique().tolist())
maxab_brand_set = set(maxab_brands)
maxab_brands_by_length = sorted(maxab_brands, key=len, reverse=True)
maxab_products = maxab.drop_duplicates(subset='PRODUCT_ID')[['PRODUCT_ID','SKU','BRAND','CAT','sku_norm','brand_norm','cat_norm']].copy().set_index('PRODUCT_ID')
brand_to_products = maxab.groupby('brand_norm')['PRODUCT_ID'].apply(set).to_dict()

# Category vocabulary with AWN expansion
print("Building category vocabulary with AWN synonyms...")
STOPWORDS = {'و','في','من','مع','بال','عرض','جنيه','زياده','مجانا','هديه','مسعر',
             'جم','كجم','مل','لتر','قطعه','حبه','كيس','ظرف','فتله','بكره',
             'خصم','قطعة','عبوة','حبة','علبة','صغير','كبير','جنية'}
cat_vocabulary = {}
for cat_norm in maxab['cat_norm'].unique():
    cat_skus = maxab[maxab['cat_norm'] == cat_norm].drop_duplicates(subset='PRODUCT_ID')
    cat_token_counts = Counter()
    for _, row in cat_skus.iterrows():
        for t in row['sku_norm'].split():
            if len(t) >= 3 and t not in STOPWORDS and not t.isdigit(): cat_token_counts[t] += 1
    cat_tokens = set(t for t in cat_norm.split() if len(t) >= 3 and t not in STOPWORDS)
    n_skus = len(cat_skus)
    if n_skus > 0:
        for token, count in cat_token_counts.items():
            if count / n_skus >= 0.08 and token not in maxab_brand_set: cat_tokens.add(token)
    for t in list(cat_tokens):
        for s in get_awn_synonyms(t):
            clean = normalize_arabic(s)
            if clean and len(clean) >= 3: cat_tokens.add(clean)
    cat_vocabulary[cat_norm] = cat_tokens
print(f"Built {len(cat_vocabulary)} category vocabularies")

# Normalize existing mapping
existing_map['scrapped_norm'] = existing_map['SCRAPPED_PRODUCT_NAME_AR'].apply(normalize_arabic)

print(f"\nReady. Maxab: {maxab['PRODUCT_ID'].nunique()} products, Scrapped: {len(scrapped)} rows, Existing map: {len(existing_map)} rows")

Building category vocabulary with AWN synonyms...
Built 70 category vocabularies

Ready. Maxab: 3811 products, Scrapped: 7435 rows, Existing map: 4792 rows


## Matching Functions (brand, category, variant, descriptive, price)

In [17]:
def category_compatible(cat_norm, scrapped_text_norm, threshold=75):
    if not cat_norm or not scrapped_text_norm: return True
    cat_tokens = cat_vocabulary.get(cat_norm, set())
    if not cat_tokens: cat_tokens = set(t for t in cat_norm.split() if len(t) >= 3 and t not in STOPWORDS)
    if not cat_tokens: return True
    scrapped_tokens = [t for t in scrapped_text_norm.split() if len(t) >= 3]
    for ct in cat_tokens:
        for st in scrapped_tokens:
            st_stripped = strip_al(st)
            if ct == st or ct == st_stripped: return True
            if fuzz.ratio(ct, st) >= threshold or fuzz.ratio(ct, st_stripped) >= threshold: return True
    return False

VARIANT_WORDS = {
    'برتقال','تفاح','مانجو','جوافه','عنب','فراوله','موز','اناناس','كوكتيل',
    'ليمون','كريز','توت','خوخ','مشمش','رمان','كمثري','بطيخ','تين',
    'كراميل','فانيليا','شوكولاته','شيكولاته','بندق','فول سوداني','جوز الهند',
    'لافندر','صنوبر','ورد','ياسمين','مسك','عود','عنبر',
    'اخضر','احمر','ازرق','اصفر','ابيض','اسود','وردي','موف','فضي','ذهبي','رمادي','زهري',
    'شيدر','رومي','فيتا','موتزاريلا','بسطرمه','قريش','جولد','استامبولي',
    'سلامي','هوت دوج','برجر','بسطرمه','لانشون','بيفي',
    'مصاصه','مارشملو','اكلير','لولي','ويفر','بسكويت','كرواسون','كيك',
    'فويل','الومنيوم','اكياس','قمامه','ارضيات','زجاج','حمام','مطبخ',
    'حلاوه','طحينه','عسل','مربي',
    'شعريه','خواتم','اسباجيتي','لسان','عصفور','هلاليه','مرمريه','لازانيا','فوسيلي','بنه',
    'نعناع','يانسون','كركديه','شاي','قرفه',
    'سائل','بودر','جيل','سبراي','كريم','صابون','رول','مناديل',
    'كمون','فلفل','كاري','بابريكا','زعتر','كركم',
    'شاور','زنجبيل','جنزبيل','اقراص',
}
def find_variants_in_text(tokens, brand_tokens):
    found = set()
    for t in tokens:
        if t in brand_tokens: continue
        t_stripped = strip_al(t)
        if t in VARIANT_WORDS: found.add(t)
        elif t_stripped in VARIANT_WORDS: found.add(t_stripped)
    return found
def variant_conflict(sn, mn, bn):
    ts, tm = sn.split(), mn.split()
    bt = set(bn.split()) if bn else set()
    vs, vm = find_variants_in_text(ts, bt), find_variants_in_text(tm, bt)
    if not vs or not vm: return None
    if vs & vm: return False
    return True

NOISE_TOKENS = STOPWORDS | {'بطعم','برائحه','محشو','بكريمه','مغطس','مغلف',
    'خاص','سعر','ع','ق','متر','باكو','باكت','جرام','كيلو',
    'بلاستيك','زجاج','حديد','صفيح','نص','ربع'}
def get_descriptive_tokens(text_norm, brand_norm):
    bt = set(brand_norm.split()) if brand_norm else set()
    return set(t for t in text_norm.split() if t not in bt and t not in NOISE_TOKENS and not t.isdigit() and len(t) >= 3 and not re.match(r'^\d+', t))
def descriptive_overlap(sn, mn, bn):
    ds, dm = get_descriptive_tokens(sn, bn), get_descriptive_tokens(mn, bn)
    if not ds or not dm: return None
    es = set()
    for t in ds:
        es.add(strip_al(t))
        for syn in get_awn_synonyms(strip_al(t)): es.add(normalize_arabic(syn))
    em = set()
    for t in dm:
        em.add(strip_al(t))
        for syn in get_awn_synonyms(strip_al(t)): em.add(normalize_arabic(syn))
    if es & em: return True
    for st in es:
        for mt in em:
            if fuzz.ratio(st, mt) >= 80: return True
    return False

CATEGORY_WORDS = {
    'مكرونه','بسكويت','جبنه','حفاضات','مناديل','نعناع','بن','مربي','كريم','لبان','ارز',
    'ملح','تونه','صابون','جيل','مسحوق','سائل','شامبو','شاي','عصير','زيت','لبن','زبادي',
    'زبده','سمن','سمنه','دقيق','سكر','شوكولاته','شيكولاته','كاكاو','قهوه','كابتشينو',
    'كوفي','شيبسي','ويفر','كيك','كورن','نودلز','صلصه','صوص','كاتشب','مايونيز','توابل',
    'فلفل','كمون','عسل','حلاوه','طحينه','طحينيه','فول','حمص','عدس','لوبيا','فاصوليا',
    'بسله','باميه','ملوخيه','سردين','ماكريل','برجر','لانشون','سجق','هوت','ناجتس','بطاطس',
    'كفته','دجاج','لحم','سمك','كروت','شحن','بطاريات','بطاريه','كبريت','فوط','مبلله',
    'مشروب','مياه','حشريه','مبيدات','منظف','منعم','مزيل','مبيضات','معطرات','معجون',
    'شاور','صبغه','حنه','ليفه','رول','كيس','علبه','كارت','حلوي','حلويات','بقسماط',
    'فريك','برغل','نشا','خميره','فانيليا','مرقه','سبريد','كاسترد','كسترد','جيلي','تمر',
    'شطه','خضار','مستلزمات','ادوات','حليب',
}
def match_brand_tawfeer(bn):
    if not bn: return []
    if bn in maxab_brand_set: return [(bn, 100)]
    results = []
    st = set(bn.split())
    for b in maxab_brands:
        bt = set(b.split())
        if st.issubset(bt) or bt.issubset(st): results.append((b, 95))
    if not results:
        best = process.extract(bn, maxab_brands, scorer=fuzz.ratio, limit=3)
        for b, s, _ in best:
            if s >= 90: results.append((b, s))
    return results
def find_brand_in_text(ptn):
    if not ptn: return []
    all_tokens = ptn.split()
    if not all_tokens: return []
    max_start = 1 if all_tokens[0] in CATEGORY_WORDS else 0
    results = []
    for brand in maxab_brands_by_length:
        bt = brand.split(); n = len(bt)
        if n == 1 and len(bt[0]) <= 2: continue
        for start in range(max_start + 1):
            if start + n > len(all_tokens): continue
            seg = all_tokens[start:start+n]
            if all(b == s for b, s in zip(bt, seg)):
                results.append((brand, 100 if start == 0 else 95)); break
            if n >= 2 and start == 0:
                if fuzz.ratio(bt[0], all_tokens[0]) >= 85:
                    if all(any(fuzz.ratio(b, all_tokens[i]) >= 85 for i in range(min(n+1, len(all_tokens)))) for b in bt):
                        results.append((brand, 90)); break
    if not results:
        for pos in range(max_start + 1):
            if pos < len(all_tokens) and len(all_tokens[pos]) >= 3 and all_tokens[pos] in maxab_brand_set:
                results.append((all_tokens[pos], 90 if pos == 0 else 85)); break
    if len(results) > 1:
        bts = [(b, set(b.split()), s) for b, s in results]
        f = [(b, s) for b, bt, s in bts if not any(bt < o for _, o, _ in bts if o != bt)]
        results = f if f else results
    return results[:5]

def price_compat(mr, sr):
    mp, mu = mr['CURRENT_PRICE'], mr['BASIC_UNIT_COUNT']
    mpu = mp / mu if mu > 0 else mp
    mut = mr['unit_canonical']
    sp = sr['SCRAPPED_PRICE']
    sq = sr.get('QUANTITY_PER_UNIT', np.nan)
    spu = sp / sq if pd.notna(sq) and sq > 0 else np.nan
    sut = sr['unit_canonical']
    has_qty = pd.notna(sq) and sq > 0
    app = sr.get('SOURCE_APP', '')
    c = []
    if mut == sut and mp > 0: c.append((pct_diff(mp, sp), 'direct'))
    if pd.notna(spu) and mpu > 0: c.append((pct_diff(mpu, spu), 'per_unit'))
    if mut == sut and has_qty and mu > 0 and mu == sq and mp > 0: c.append((pct_diff(mp, sp), 'same_qty'))
    if mpu > 0 and sut in ('piece', 'item') and has_qty: c.append((pct_diff(mpu, sp), 'scrapped_piece'))
    if mu == 1 and has_qty and sq > 1 and spu > 0 and mp > 0: c.append((pct_diff(mp, spu), 'maxab_piece'))
    if not has_qty and app in NO_QTY_APPS and mp > 0: c.append((pct_diff(mp, sp), 'blind_direct'))
    if not c: return False, 1.0, 'none'
    best = min(c, key=lambda x: x[0])
    return best[0] <= 0.20, best[0], best[1]

print("Matching functions loaded.")

Matching functions loaded.


## Step 1: Validate Existing Manual Mapping

In [18]:
def validate_existing_match(product_id, scrapped_product_norm, source_app):
    """Validate a manually-mapped pair using our filters.
    Returns (valid, reason) tuple."""
    if product_id not in maxab_products.index:
        return False, 'product_not_in_maxab'
    
    row = maxab_products.loc[product_id]
    
    # Size check
    sz = sizes_compatible(scrapped_product_norm, row['sku_norm'], 0.15)
    if sz is False:
        return False, 'size_mismatch'
    
    # Variant conflict
    vc = variant_conflict(scrapped_product_norm, row['sku_norm'], row['brand_norm'])
    if vc is True:
        return False, 'variant_conflict'
    
    # Text similarity (loose threshold for manual mapping -- they had human judgment)
    ts = text_sim(scrapped_product_norm, row['sku_norm'])
    if ts < 35:
        return False, f'text_sim_too_low_{ts:.0f}'
    
    return True, f'valid_ts{ts:.0f}'

print("Validating existing manual mapping...")
validated = []
rejected = []

for _, em in existing_map.iterrows():
    pid = em['MAXAB_PRODUCT_ID']
    sn = em['scrapped_norm']
    app = em['SOURCE_APP']
    
    valid, reason = validate_existing_match(pid, sn, app)
    
    if valid:
        validated.append({
            'PRODUCT_ID': pid,
            'SOURCE_APP': app,
            'SCRAPPED_PRODUCT': em['SCRAPPED_PRODUCT_NAME_AR'],
            'scrapped_norm': sn,
            'match_source': 'existing_mapping',
            'validation_reason': reason,
        })
    else:
        rejected.append({
            'PRODUCT_ID': pid,
            'SOURCE_APP': app,
            'SCRAPPED_PRODUCT': em['SCRAPPED_PRODUCT_NAME_AR'],
            'rejection_reason': reason,
        })

validated_df = pd.DataFrame(validated)
rejected_df = pd.DataFrame(rejected)

print(f"Existing mapping: {len(existing_map)} total")
print(f"  Validated: {len(validated_df)} ({100*len(validated_df)/len(existing_map):.1f}%)")
print(f"  Rejected:  {len(rejected_df)} ({100*len(rejected_df)/len(existing_map):.1f}%)")
if len(rejected_df):
    print(f"\n  Rejection reasons:")
    print(f"  {rejected_df['rejection_reason'].value_counts().to_string()}")
print(f"\n  Validated unique maxab products: {validated_df['PRODUCT_ID'].nunique()}")

Validating existing manual mapping...
Existing mapping: 4792 total
  Validated: 2549 (53.2%)
  Rejected:  2243 (46.8%)

  Rejection reasons:
  rejection_reason
product_not_in_maxab    1805
variant_conflict         342
size_mismatch             94
text_sim_too_low_23        1
text_sim_too_low_27        1

  Validated unique maxab products: 1724


## Step 2: Run Algorithm on Unmapped Products

In [19]:
# Build set of already-mapped (app, scrapped_product) pairs from validated existing mapping
already_mapped = set()
if len(validated_df):
    for _, v in validated_df.iterrows():
        already_mapped.add((v['SOURCE_APP'], v['scrapped_norm']))

# Filter scrapped data to only unmapped products
scrapped_unmapped = scrapped[~scrapped.apply(lambda r: (r['SOURCE_APP'], r['product_norm']) in already_mapped, axis=1)]
print(f"Scrapped total: {len(scrapped)}, Already mapped: {len(scrapped) - len(scrapped_unmapped)}, To process: {len(scrapped_unmapped)}")

def get_candidates(sr, top_n=10):
    app, pn, bn = sr['SOURCE_APP'], sr['product_norm'], sr['brand_norm']
    cids, bsm = set(), {}
    if app == 'Tawfeer' and bn:
        for mb, sc in match_brand_tawfeer(bn):
            if mb in brand_to_products:
                for p in brand_to_products[mb]: cids.add(p); bsm[p] = max(bsm.get(p, 0), sc)
    else:
        for mb, sc in find_brand_in_text(pn):
            if mb in brand_to_products:
                for p in brand_to_products[mb]: cids.add(p); bsm[p] = max(bsm.get(p, 0), sc)
    if not cids: return []
    st = f"{bn} {pn}" if app == 'Tawfeer' and bn else pn
    bw = 0.15 if (app == 'Tawfeer' and bn) else 0.05
    scored = []
    for pid in cids:
        if pid not in maxab_products.index: continue
        row = maxab_products.loc[pid]
        ts = text_sim(st, row['sku_norm'])
        cat_ok = category_compatible(row['cat_norm'], pn)
        if not cat_ok and ts < SOFT_THRESHOLD: continue
        sz = sizes_compatible(st, row['sku_norm'], 0.15)
        if sz is False: continue
        if ts < SOFT_THRESHOLD:
            desc_ok = descriptive_overlap(pn, row['sku_norm'], row['brand_norm'])
            if desc_ok is False: continue
        vc = variant_conflict(pn, row['sku_norm'], row['brand_norm'])
        if vc is True: continue
        scored.append((pid, ts + bsm.get(pid, 0) * bw + (10 if sz is True else 0), ts, bsm.get(pid, 0)))
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_n]

def match_units(mrows, sr, text_similarity):
    results = []
    for _, m in mrows.iterrows():
        um = m['unit_canonical'] == sr['unit_canonical']
        po, pd_, ps = price_compat(m, sr)
        if pd_ > PRICE_HARD_MAX: continue
        if not po and text_similarity < 65: continue
        us = (30 if um else 0) + (max(0, 30 * (1 - pd_)) if po else 0)
        results.append({'PRODUCT_ID': m['PRODUCT_ID'], 'SKU': m['SKU'], 'BRAND': m['BRAND'], 'CAT': m['CAT'],
            'CURRENT_PRICE': m['CURRENT_PRICE'], 'BASIC_UNIT_COUNT': m['BASIC_UNIT_COUNT'],
            'PACKING_UNIT_NAME_EN': m['PACKING_UNIT_NAME_EN'], 'PACKING_UNIT_NAME_AR': m['PACKING_UNIT_NAME_AR'],
            'unit_match': um, 'price_compatible': po, 'price_diff_pct': pd_, 'price_strategy': ps, 'unit_price_score': us})
    results.sort(key=lambda x: x['unit_price_score'], reverse=True)
    return results

def map_one(sr, min_ts=45):
    cands = get_candidates(sr)
    if not cands: return []
    matches = []
    for pid, cs, ts, bs in cands:
        if ts < min_ts: continue
        for um in match_units(maxab[maxab['PRODUCT_ID'] == pid], sr, ts):
            um.update({'text_similarity': ts, 'brand_similarity': bs, 'combined_score': cs + um['unit_price_score'],
                'SOURCE_APP': sr['SOURCE_APP'], 'SUPPLIER': sr['SUPPLIER'], 'SCRAPPED_PRODUCT': sr['SCRAPPED_PRODUCT'],
                'SCRAPPED_BRAND': sr.get('SCRAPPED_BRAND', ''), 'SCRAPPED_PRICE': sr['SCRAPPED_PRICE'],
                'QUANTITY_PER_UNIT': sr.get('QUANTITY_PER_UNIT', np.nan), 'SCRAPPED_UNIT_TYPE': sr['UNIT_TYPE'],
                'match_source': 'algorithm'})
            matches.append(um)
    matches.sort(key=lambda x: x['combined_score'], reverse=True)
    return matches

print(f"\nRunning algorithm on {len(scrapped_unmapped)} unmapped products...")
print(f"Started at {datetime.now().strftime('%H:%M:%S')}")
algo_results, algo_unmatched = [], []
mc = {'t': 0, 'm': 0}
for idx, sr in scrapped_unmapped.iterrows():
    ms = map_one(sr)
    mc['t'] += 1
    if ms:
        algo_results.append(ms[0]); mc['m'] += 1
        for m in ms[1:]:
            if m['combined_score'] >= ms[0]['combined_score'] * 0.90: algo_results.append(m)
    else:
        algo_unmatched.append({'SOURCE_APP': sr['SOURCE_APP'], 'SCRAPPED_PRODUCT': sr['SCRAPPED_PRODUCT'],
            'SCRAPPED_BRAND': sr.get('SCRAPPED_BRAND', ''), 'SCRAPPED_PRICE': sr['SCRAPPED_PRICE'], 'SCRAPPED_UNIT_TYPE': sr['UNIT_TYPE']})
    if (mc['t']) % 500 == 0: print(f"  {mc['t']}/{len(scrapped_unmapped)} ({mc['m']} matched)")

algo_df = pd.DataFrame(algo_results)
algo_unmatched_df = pd.DataFrame(algo_unmatched)
print(f"\nDone at {datetime.now().strftime('%H:%M:%S')}")
print(f"Algorithm: {mc['m']}/{mc['t']} matched ({100*mc['m']/mc['t']:.1f}%), {len(algo_df)} result rows")

Scrapped total: 7435, Already mapped: 1189, To process: 6246

Running algorithm on 6246 unmapped products...
Started at 14:06:21
  500/6246 (129 matched)
  1000/6246 (248 matched)
  1500/6246 (349 matched)
  2000/6246 (466 matched)
  2500/6246 (582 matched)
  3000/6246 (693 matched)
  3500/6246 (795 matched)
  4000/6246 (904 matched)
  4500/6246 (1009 matched)
  5000/6246 (1130 matched)
  5500/6246 (1244 matched)
  6000/6246 (1360 matched)

Done at 14:08:35
Algorithm: 1414/6246 matched (22.6%), 3493 result rows


## Step 3: Combine + Compare

In [20]:
# Combine: validated existing + algorithm results
existing_products = set(validated_df['PRODUCT_ID'].unique()) if len(validated_df) else set()
algo_products = set(algo_df['PRODUCT_ID'].unique()) if len(algo_df) else set()
combined_products = existing_products | algo_products
algo_only = algo_products - existing_products
existing_only = existing_products - algo_products
overlap = existing_products & algo_products

print("=== COMPARISON ===")
print(f"\n  Existing mapping (validated):  {len(existing_products)} unique maxab products")
print(f"  Algorithm alone:               {algo_products.__len__()} unique maxab products")
print(f"  Combined:                      {len(combined_products)} unique maxab products")
print(f"  Total maxab products:          {maxab['PRODUCT_ID'].nunique()}")
print(f"  Combined coverage:             {100*len(combined_products)/maxab['PRODUCT_ID'].nunique():.1f}%")
print(f"\n  Overlap (both found):          {len(overlap)}")
print(f"  Only in existing mapping:      {len(existing_only)}")
print(f"  Only found by algorithm:       {len(algo_only)}")

print(f"\n  Existing mapping rejected:     {len(rejected_df)} rows")
if len(rejected_df):
    print(f"  Rejection breakdown:")
    print(f"  {rejected_df['rejection_reason'].value_counts().head(10).to_string()}")

# Per-app breakdown
print(f"\n=== Per-App Coverage ===")
for app in sorted(scrapped['SOURCE_APP'].unique()):
    ex_count = validated_df[validated_df['SOURCE_APP']==app]['PRODUCT_ID'].nunique() if len(validated_df) else 0
    al_count = algo_df[algo_df['SOURCE_APP']==app]['PRODUCT_ID'].nunique() if len(algo_df) else 0
    combined = len(set(validated_df[validated_df['SOURCE_APP']==app]['PRODUCT_ID'].unique()) | set(algo_df[algo_df['SOURCE_APP']==app]['PRODUCT_ID'].unique())) if len(validated_df) or len(algo_df) else 0
    print(f"  {app}: existing={ex_count}, algorithm={al_count}, combined={combined}")

=== COMPARISON ===

  Existing mapping (validated):  1724 unique maxab products
  Algorithm alone:               973 unique maxab products
  Combined:                      1958 unique maxab products
  Total maxab products:          3811
  Combined coverage:             51.4%

  Overlap (both found):          739
  Only in existing mapping:      985
  Only found by algorithm:       234

  Existing mapping rejected:     2243 rows
  Rejection breakdown:
  rejection_reason
product_not_in_maxab    1805
variant_conflict         342
size_mismatch             94
text_sim_too_low_23        1
text_sim_too_low_27        1

=== Per-App Coverage ===
  Cartona: existing=1119, algorithm=309, combined=1235
  Speed: existing=414, algorithm=299, combined=581
  Talabia: existing=587, algorithm=290, combined=732
  Tawfeer: existing=429, algorithm=401, combined=552


## Export

In [22]:
export_cols = ['SOURCE_APP','SUPPLIER','SCRAPPED_PRODUCT','SCRAPPED_BRAND','SCRAPPED_PRICE',
    'QUANTITY_PER_UNIT','SCRAPPED_UNIT_TYPE','PRODUCT_ID','SKU','BRAND','CAT','CURRENT_PRICE',
    'BASIC_UNIT_COUNT','PACKING_UNIT_NAME_AR','PACKING_UNIT_NAME_EN',
    'text_similarity','brand_similarity','price_compatible','price_diff_pct',
    'price_strategy','combined_score','match_source']

if len(algo_df):
    algo_export = algo_df[[c for c in export_cols if c in algo_df.columns]].sort_values(['SOURCE_APP','combined_score'], ascending=[True,False])
    algo_export.to_excel(r'mapping_algorithm_results.xlsx', index=False)
    print(f"Algorithm results: {len(algo_export)} rows exported")

if len(validated_df):
    validated_df.to_excel(r'mapping_existing_validated.xlsx', index=False)
    print(f"Validated existing mapping: {len(validated_df)} rows exported")

if len(rejected_df):
    rejected_df.to_excel(r'mapping_existing_rejected.xlsx', index=False)
    print(f"Rejected existing mapping: {len(rejected_df)} rows exported")

if len(algo_unmatched_df):
    algo_unmatched_df.to_excel(r'mapping_unmatched.xlsx', index=False)
    print(f"Unmatched: {len(algo_unmatched_df)} rows exported")

# Combined file: existing validated + algorithm results
combined_rows = []

# Add validated existing mapping with maxab product info
if len(validated_df):
    for _, v in validated_df.iterrows():
        pid = v['PRODUCT_ID']
        if pid in maxab_products.index:
            mp = maxab_products.loc[pid]
            combined_rows.append({
                'SOURCE_APP': v['SOURCE_APP'],
                'SCRAPPED_PRODUCT': v['SCRAPPED_PRODUCT'],
                'PRODUCT_ID': pid,
                'SKU': mp['SKU'],
                'BRAND': mp['BRAND'],
                'CAT': mp['CAT'],
                'match_source': 'existing_mapping',
            })

# Add algorithm results
if len(algo_df):
    for _, a in algo_df.iterrows():
        combined_rows.append({
            'SOURCE_APP': a['SOURCE_APP'],
            'SUPPLIER': a.get('SUPPLIER', ''),
            'SCRAPPED_PRODUCT': a['SCRAPPED_PRODUCT'],
            'SCRAPPED_BRAND': a.get('SCRAPPED_BRAND', ''),
            'SCRAPPED_PRICE': a['SCRAPPED_PRICE'],
            'QUANTITY_PER_UNIT': a.get('QUANTITY_PER_UNIT', np.nan),
            'SCRAPPED_UNIT_TYPE': a.get('SCRAPPED_UNIT_TYPE', ''),
            'PRODUCT_ID': a['PRODUCT_ID'],
            'SKU': a['SKU'],
            'BRAND': a['BRAND'],
            'CAT': a['CAT'],
            'CURRENT_PRICE': a['CURRENT_PRICE'],
            'BASIC_UNIT_COUNT': a['BASIC_UNIT_COUNT'],
            'PACKING_UNIT_NAME_AR': a.get('PACKING_UNIT_NAME_AR', ''),
            'PACKING_UNIT_NAME_EN': a.get('PACKING_UNIT_NAME_EN', ''),
            'text_similarity': a.get('text_similarity', np.nan),
            'price_diff_pct': a.get('price_diff_pct', np.nan),
            'price_strategy': a.get('price_strategy', ''),
            'match_source': 'algorithm',
        })

combined_df = pd.DataFrame(combined_rows)
combined_df.to_excel(r'd:\scripts\Mustafa\Mapping\mapping_combined.xlsx', index=False)
print(f"Combined (existing + algorithm): {len(combined_df)} rows, {combined_df['PRODUCT_ID'].nunique()} unique maxab products")
print(f"  From existing: {(combined_df['match_source']=='existing_mapping').sum()} rows")
print(f"  From algorithm: {(combined_df['match_source']=='algorithm').sum()} rows")

print("\nAll files saved to Mapping folder.")

Algorithm results: 3493 rows exported
Validated existing mapping: 2549 rows exported
Rejected existing mapping: 2243 rows exported
Unmatched: 4832 rows exported

All files saved to Mapping folder.
